## OpenNSFW2 moderation model


In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from PIL import Image
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

import opennsfw2 as n2
import tensorflow as tf

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts" / "notebooks"
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid")


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


def load_image_frame() -> pd.DataFrame:
    for candidate in [Path(path) for path in DATASET_CANDIDATES]:
        candidate = candidate if candidate.is_absolute() else (NOTEBOOK_DIR / candidate).resolve()
        if not candidate.exists():
            continue
        if candidate.suffix == ".parquet":
            frame = pd.read_parquet(candidate)
        else:
            frame = pd.read_csv(candidate)
        break
    else:
        raise FileNotFoundError("No image manifest was found.")

    if "image_path" not in frame.columns:
        path_like = [col for col in frame.columns if "path" in col.lower()]
        frame["image_path"] = frame[path_like[0]]
    if "label" not in frame.columns:
        label_like = [col for col in frame.columns if "label" in col.lower() or "nsfw" in col.lower()]
        frame["label"] = frame[label_like[0]] if label_like else np.where(frame.index % 2 == 0, 0, 1)
    frame["label"] = pd.to_numeric(frame["label"], errors="coerce").fillna(0).astype(int)
    frame = frame[frame["image_path"].map(lambda value: Path(str(value)).exists())].copy()
    return frame[["image_path", "label"]].reset_index(drop=True)


image_df = load_image_frame()
train_df, valid_df = train_test_split(
    image_df,
    test_size=0.2 if len(image_df) >= 50 else 0.3,
    stratify=image_df["label"] if image_df["label"].nunique() > 1 else None,
    random_state=42,
)
model = n2.make_open_nsfw_model()
model.summary()


In [ ]:
def preprocess_with_mode(path: str, preprocessing) -> np.ndarray:
    image = Image.open(path).convert("RGB")
    return n2.preprocess_image(image, preprocessing=preprocessing)


sample_paths = valid_df["image_path"].head(min(6, len(valid_df))).tolist()
yahoo_batch = np.stack([preprocess_with_mode(path, n2.Preprocessing.YAHOO) for path in sample_paths], axis=0)
simple_batch = np.stack([preprocess_with_mode(path, n2.Preprocessing.SIMPLE) for path in sample_paths], axis=0)

yahoo_scores = model.predict(yahoo_batch, verbose=0)[:, 1]
simple_scores = model.predict(simple_batch, verbose=0)[:, 1]
pd.DataFrame({"path": sample_paths, "score_yahoo": yahoo_scores, "score_simple": simple_scores})


In [ ]:
penultimate = tf.keras.Model(model.input, model.layers[-2].output)
activations = penultimate.predict(yahoo_batch, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.heatmap(pd.DataFrame(activations[: min(len(activations), 20)]).T.iloc[:64], cmap="mako", ax=axes[0])
sns.histplot(activations.flatten(), bins=40, ax=axes[1])
plt.tight_layout()


In [ ]:
def compute_binary_metrics(y_true, scores, threshold: float = 0.5) -> dict:
    y_true = np.asarray(y_true).astype(int)
    scores = np.asarray(scores, dtype=float)
    pred = (scores >= threshold).astype(int)
    tp = int(((pred == 1) & (y_true == 1)).sum())
    tn = int(((pred == 0) & (y_true == 0)).sum())
    fp = int(((pred == 1) & (y_true == 0)).sum())
    fn = int(((pred == 0) & (y_true == 1)).sum())
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-8)
    fpr = fp / max(fp + tn, 1)
    try:
        auc = roc_auc_score(y_true, scores)
    except ValueError:
        auc = float("nan")
    return {
        "Precision": round(float(precision), 4),
        "Recall": round(float(recall), 4),
        "F1": round(float(f1), 4),
        "FPR": round(float(fpr), 4),
        "AUC": round(float(auc), 4) if not np.isnan(auc) else None,
        "threshold": threshold,
    }


    def extract_features(paths: list[str], preprocessing=n2.Preprocessing.YAHOO) -> np.ndarray:
        batch = np.stack([preprocess_with_mode(path, preprocessing) for path in paths], axis=0)
        return penultimate.predict(batch, verbose=0)


    train_features = extract_features(train_df["image_path"].tolist())
    valid_features = extract_features(valid_df["image_path"].tolist())

    base_clf = LogisticRegression(max_iter=4000, class_weight="balanced")
    base_clf.fit(train_features, train_df["label"])
    valid_scores = base_clf.predict_proba(valid_features)[:, 1]
    base_metrics = compute_binary_metrics(valid_df["label"], valid_scores, threshold=0.5)
    base_metrics


In [ ]:
class LoRAHead(tf.keras.Model):
    def __init__(self, input_dim: int, rank: int = 8, use_dora: bool = False):
        super().__init__()
        self.base = tf.keras.layers.Dense(2)
        self.a = tf.keras.layers.Dense(rank, use_bias=False)
        self.b = tf.keras.layers.Dense(2, use_bias=False)
        self.scale = 16.0 / rank
        self.use_dora = use_dora
        if use_dora:
            self.magnitude = tf.Variable(tf.ones((2,)), trainable=True)

    def call(self, inputs):
        outputs = self.base(inputs) + self.b(self.a(inputs)) * self.scale
        if self.use_dora:
            outputs = outputs * self.magnitude
        return outputs


def train_adapter(use_dora: bool, run_name: str):
    adapter = LoRAHead(train_features.shape[1], rank=8, use_dora=use_dora)
    adapter.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=[tf.keras.metrics.Precision(name="precision"), tf.keras.metrics.Recall(name="recall")],
    )
    run_dir = ARTIFACT_ROOT / run_name
    run_dir.mkdir(parents=True, exist_ok=True)
    adapter.fit(
        train_features,
        train_df["label"].to_numpy(),
        validation_data=(valid_features, valid_df["label"].to_numpy()),
        epochs=3,
        batch_size=32,
        verbose=0,
    )
    logits = adapter.predict(valid_features, verbose=0)
    probs = tf.nn.softmax(logits, axis=-1).numpy()[:, 1]
    metrics = compute_binary_metrics(valid_df["label"], probs, threshold=0.5)
    adapter.save_weights(run_dir / "weights")
    return metrics


lora_metrics = train_adapter(use_dora=False, run_name="opennsfw2_lora_head")
dora_metrics = train_adapter(use_dora=True, run_name="opennsfw2_dora_head")
pd.DataFrame([{"run": "base", **base_metrics}, {"run": "lora", **lora_metrics}, {"run": "dora", **dora_metrics}])
